In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Semantic search

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

C:\Users\dell\AppData\Local\Temp\ipykernel_18040\4167041359.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
incorrect startxref pointer(1)
parsing for Object Streams


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'resources/acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

3


Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

In [4]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [5]:
embeddings

OllamaEmbeddings(model='nomic-embed-text', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [7]:
ids = vector_store.add_documents(documents=all_splits)

In [8]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

## RAG Agent

In [9]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [10]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOllama(model="llama3.2")

agent = create_agent(
    model=llm,
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information.",
    checkpointer=InMemorySaver()
)

In [13]:
from langchain.messages import HumanMessage
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}, 
    config=config
)

In [11]:
from pprint import pprint

pprint(response["messages"][-1].content)

('In your first year (0–1 year of service), you get 10 PTO days per year '
 '(about 0.833 days per month). You can carry over up to 5 unused PTO days per '
 'calendar year.')


In [12]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Does Travel policy change if an employee takes more holidays?")]}, 
    config=config
)

In [13]:

pprint(response["messages"][-1].content)

('No. The Travel & Expense Policy does not change based on how many holidays '
 'you take.\n'
 '\n'
 '- The policy covers reimbursing reasonable and necessary expenses for '
 'approved business travel within established limits.\n'
 '- PTO or holidays are separate from travel reimbursements.\n'
 '- If you’re not on approved business travel (e.g., you’re on vacation), '
 'there are no travel expenses to claim.\n'
 '- If you do have travel, remember to submit receipts within 14 days of '
 'travel.\n'
 '- Extended absences longer than 5 consecutive business days require manager '
 'approval.')


In [15]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is the company's policy on remote work?")]}, 
    config=config
)

pprint(response["messages"][-1].content)

('There isn’t an explicit remote work policy in the handbook sections I can '
 'access. The available policies cover PTO, Travel & Expense, NDA, and '
 'Workplace Conduct, but none specify remote work eligibility, equipment, '
 'security requirements, or approval processes.\n'
 '\n'
 'If you need specifics, please check with HR or your manager for any '
 'department-specific guidance or an addendum. I can also help draft a '
 'proposed remote work policy or search for additional documents if you have '
 'access to more handbook sections.')


In [16]:
response

{'messages': [HumanMessage(content='How many days of vacation does an employee get in their first year?', additional_kwargs={}, response_metadata={}, id='49e4331d-180f-4927-9a13-c26c84cb9b11'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 157, 'total_tokens': 313, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DpXLp7QnU8jiwhSAKClNls7KqsMXY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eb652-3395-77b0-8f41-104fd93d4273-0', tool_calls=[{'name': 'search_handbook', 'args': {'query': 'vacation days first year'}, 'id': 'call_EY2xFfxQTQcdxuDaKXvymYW6', 'type': 'tool_call'}], invalid_too